# 01 — Data Preprocessing

**Pipeline version:** 2.0-consolidated  
**Purpose:** Load raw TCGA-PRAD data, clean clinical + RNA-Seq, merge, and produce
analysis-ready train/test splits.  
**Outputs:** `data/processed/X_train_preprocessed.csv`, `X_test_preprocessed.csv`,
`y_train.csv`, `y_test.csv`, `selected_features_final.csv`  

All heavy numerical preprocessing (scaling, log, winsorize) is deferred to
inside CV folds to prevent leakage.  This notebook only produces clean,
merged DataFrames.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name != "core" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import config
from src.io import logger, save_dataframe
from src.clinical import preprocess_clinical
from src.genomics import preprocess_rna_seq
from src.merge import merge_datasets, check_duplicates
from src.leakage import audit_leakage

logger.info("Project root: %s", project_root)
logger.info("Pipeline version: %s", config.PIPELINE_VERSION)

2026-08-26 02:25:46 | INFO     | prostate_bcr | Project root: d:\Prostate_BCR\core
2026-08-26 02:25:46 | INFO     | prostate_bcr | Pipeline version: 2.0-consolidated


## 1. Load & Clean Clinical Data

In [2]:
from src.io import load_clinical_raw

raw_clinical = load_clinical_raw()
logger.info("Raw clinical shape: %s", raw_clinical.shape)

# Run clinical preprocessing pipeline
clinical = preprocess_clinical(raw_clinical)
logger.info("Cleaned clinical shape: %s", clinical.shape)
logger.info("Target distribution:\n%s", clinical[config.TARGET_COLUMN].value_counts().to_string())

2026-08-26 02:25:46 | INFO     | prostate_bcr | Loading raw clinical data: D:\Prostate_BCR\core\data\raw\data_clinical_patient.tsv
2026-08-26 02:25:46 | INFO     | prostate_bcr | Loaded clinical data: 504 rows × 69 columns
2026-08-26 02:25:46 | INFO     | prostate_bcr | Raw clinical shape: (504, 69)
2026-08-26 02:25:46 | INFO     | prostate_bcr | Dropped 4 metadata rows → 500 patients
2026-08-26 02:25:46 | INFO     | prostate_bcr | Detected 12 numeric / 56 categorical columns
2026-08-26 02:25:46 | INFO     | prostate_bcr | Dropped 21 near-constant columns
2026-08-26 02:25:46 | INFO     | prostate_bcr | Removed 7 leakage columns
2026-08-26 02:25:46 | INFO     | prostate_bcr | Dropped #Other Patient ID (500 levels > 15)
2026-08-26 02:25:46 | INFO     | prostate_bcr | Dropped Form completion date (128 levels > 15)
2026-08-26 02:25:46 | INFO     | prostate_bcr | Dropped Days to bone Scan performed (92 levels > 15)
2026-08-26 02:25:46 | INFO     | prostate_bcr | Dropped Days to ct scan ab p

## 2. Load & Clean RNA-Seq Data

In [3]:
rna_seq = preprocess_rna_seq()
logger.info("Cleaned RNA-Seq shape: %s", rna_seq.shape)

2026-08-26 02:25:46 | INFO     | prostate_bcr | Loading RNA-Seq data from D:\Prostate_BCR\core\data\raw\data_mrna_seq_v2_rsem.txt
2026-08-26 02:25:47 | INFO     | prostate_bcr | Dropped metadata columns: ['Entrez_Gene_Id']
2026-08-26 02:25:47 | INFO     | prostate_bcr | Loaded RNA-Seq matrix: 20531 genes × 498 samples
2026-08-26 02:25:47 | INFO     | prostate_bcr | Found 17 duplicate gene names
2026-08-26 02:25:47 | INFO     | prostate_bcr | Resolved duplicates → 20514 unique genes
2026-08-26 02:25:47 | INFO     | prostate_bcr | Filtered low-expression genes: 20514 → 18905 (removed 1609)
2026-08-26 02:25:47 | INFO     | prostate_bcr | Removed 1 duplicate patients from index
2026-08-26 02:25:47 | INFO     | prostate_bcr | Transposed and standardized: 497 samples × 18905 genes
2026-08-26 02:25:48 | INFO     | prostate_bcr | QC Report:
2026-08-26 02:25:48 | INFO     | prostate_bcr |   n_samples: 497.00
2026-08-26 02:25:48 | INFO     | prostate_bcr |   n_genes: 18905.00
2026-08-26 02:25:48

## 3. Merge & Leakage Audit

In [4]:
# Merge clinical + RNA-Seq by Patient Identifier
X_merged, y = merge_datasets(clinical, rna_seq)
logger.info("Merged: %d samples × %d features", X_merged.shape[0], X_merged.shape[1])

# Leakage audit
X_clean, leakage_report = audit_leakage(
    X_merged.assign(**{config.TARGET_COLUMN: y}),
    drop_columns=True,
    raw_clinical_path=config.CLINICAL_RAW,
)

# Separate features and target after audit
y = X_clean[config.TARGET_COLUMN].copy()
X = X_clean.drop(columns=[config.TARGET_COLUMN, config.PATIENT_ID_COLUMN], errors="ignore")

logger.info("Post-leakage audit: %d samples × %d features", X.shape[0], X.shape[1])
logger.info("Positive rate: %.1f%%", y.mean() * 100)

2026-08-26 02:25:48 | INFO     | prostate_bcr | Found 429 common patients
2026-08-26 02:25:48 | INFO     | prostate_bcr | Merge validation passed: 429 samples aligned
2026-08-26 02:25:48 | INFO     | prostate_bcr | Merged dataset: 429 samples × 19019 features (114 clinical, 18905 genes)
2026-08-26 02:25:48 | INFO     | prostate_bcr | Merged: 429 samples × 19019 features
2026-08-26 02:25:48 | INFO     | prostate_bcr | ============================================================
2026-08-26 02:25:48 | INFO     | prostate_bcr | LEAKAGE AUDIT REPORT
2026-08-26 02:25:48 | INFO     | prostate_bcr | ============================================================
2026-08-26 02:25:48 | INFO     | prostate_bcr | No known leakage columns detected
2026-08-26 02:25:48 | WARNING  | prostate_bcr | Column 'Patient Identifier' not found; skipping duplicate check
2026-08-26 02:25:48 | INFO     | prostate_bcr | Running temporal PSA audit on D:\Prostate_BCR\core\data\raw\data_clinical_patient.tsv
2026-08-26 0

## 4. Train/Test Split (stratified)

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config.TEST_SIZE,
    stratify=y,
    random_state=config.RANDOM_STATE,
)

logger.info("Train: %d (BCR+=%d, BCR−=%d)", len(y_train), int(y_train.sum()), int((y_train==0).sum()))
logger.info("Test:  %d (BCR+=%d, BCR−=%d)", len(y_test), int(y_test.sum()), int((y_test==0).sum()))

2026-08-26 02:25:49 | INFO     | prostate_bcr | Train: 343 (BCR+=46, BCR−=297)
2026-08-26 02:25:49 | INFO     | prostate_bcr | Test:  86 (BCR+=12, BCR−=74)


## 5. Save Processed Artifacts

In [6]:
save_dataframe(X_train, "X_train_preprocessed.csv")
save_dataframe(X_test, "X_test_preprocessed.csv")
save_dataframe(y_train.to_frame(name=config.TARGET_COLUMN), "y_train.csv")
save_dataframe(y_test.to_frame(name=config.TARGET_COLUMN), "y_test.csv")

# Save feature list (all features at this stage — selection happens in NB02)
pd.DataFrame({"feature": X.columns.tolist()}).to_csv(
    config.TABLES_DIR / "all_features.csv", index=False
)

logger.info("Artifacts saved to %s", config.PROCESSED_DIR)
logger.info("Done: 01_Data_Preprocessing")

2026-08-26 02:25:54 | INFO     | prostate_bcr | Saved 343 rows → D:\Prostate_BCR\core\data\processed\X_train_preprocessed.csv
2026-08-26 02:25:56 | INFO     | prostate_bcr | Saved 86 rows → D:\Prostate_BCR\core\data\processed\X_test_preprocessed.csv
2026-08-26 02:25:56 | INFO     | prostate_bcr | Saved 343 rows → D:\Prostate_BCR\core\data\processed\y_train.csv
2026-08-26 02:25:56 | INFO     | prostate_bcr | Saved 86 rows → D:\Prostate_BCR\core\data\processed\y_test.csv
2026-08-26 02:25:56 | INFO     | prostate_bcr | Artifacts saved to D:\Prostate_BCR\core\data\processed
2026-08-26 02:25:56 | INFO     | prostate_bcr | Done: 01_Data_Preprocessing
